# Flight Departure Delay (T-2h) — Person 2: Baselines, Trees & PCA

Ready-to-run notebook for **Google Colab** or **Kaggle**.

### What this notebook does
1. Sets up the repository and locates `train_T2h.csv` and `validation_T2h.csv` (never touches `test_T2h.csv`).
2. Runs the shared 4-fold expanding chronological protocol with seeds `(11, 23, 42)`.
3. Tunes on a continuous 100,000-row block inside each training fold and scores the full validation block.
4. Produces Person 4 handoff files in `person2_outputs/`.

In [ ]:
# 1. Set up train.py, person2/person2_pipeline.py & COLUMN_AUDIT.md (no git clone needed!)
from pathlib import Path

Path('person2').mkdir(exist_ok=True)
Path('person2/person2_pipeline.py').write_text('"""Person 2 modelling pipeline: Baselines, Decision Tree, Random Forest, AdaBoost, and PCA.\n\nImplements the shared T-2h evaluation protocol:\n- 4 expanding chronological validation folds (Sep 1-21, Sep 22-Oct 12, Oct 13-Nov 2, Nov 3-28)\n- Strictly in-fold preprocessing (top-20 Dest + Other, one-hot, cyclic sin/cos, median imputation, scaling)\n- Seeds (11, 23, 42) averaged within each fold, then mean +/- std across the 4 folds\n- Primary metric: Average Precision (AP / PR-AUC); Secondary: ROC-AUC\n- Optional 100,000-row continuous block for fast hyperparameter tuning, full validation scoring\n- Outputs validation score table, OOF probability predictions, PCA stats, and chosen settings for Person 4\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.decomposition import PCA\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.tree import DecisionTreeClassifier\n\nSEEDS = (11, 23, 42)\nMETA_COLS = ["row_id", "FlightDate", "scheduled_dep_utc", "checkpoint_utc", "wx_obs_time"]\nTARGET_COL = "LATE"\n\nFOLDS = [\n    (1, "2025-09-01", "2025-09-21"),\n    (2, "2025-09-22", "2025-10-12"),\n    (3, "2025-10-13", "2025-11-02"),\n    (4, "2025-11-03", "2025-11-28"),\n]\n\n# 35 raw predictors (T-2h)\nSCHEDULE_COLS = ["month", "dow", "dep_hour", "arr_hour", "CRSElapsedTime", "Distance", "is_holiday"]\nROUTE_CARRIER_COLS = ["Origin", "Dest", "Reporting_Airline"]\nCAT_COLS = ["Origin", "Dest", "Reporting_Airline", "wx_skyc1_T2h"]\n\n\ndef get_feature_columns(df: pd.DataFrame) -> list[str]:\n    """Return the 35 predictor columns in order, excluding metadata and target."""\n    excluded = set(META_COLS) | {TARGET_COL}\n    cols = [c for c in df.columns if c not in excluded]\n    return cols\n\n\ndef load_dev_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Load train_T2h.csv and validation_T2h.csv (never touches test_T2h.csv)."""\n    train_path = data_dir / "train_T2h.csv"\n    val_path = data_dir / "validation_T2h.csv"\n    if not train_path.exists() or not val_path.exists():\n        raise FileNotFoundError(\n            f"Expected train_T2h.csv and validation_T2h.csv in {data_dir}. "\n            "Run the dataset pipeline or point --data-dir to your CSV folder."\n        )\n    train_df = pd.read_csv(train_path, low_memory=False)\n    val_df = pd.read_csv(val_path, low_memory=False)\n    for frame in (train_df, val_df):\n        frame["scheduled_dep_utc"] = pd.to_datetime(frame["scheduled_dep_utc"], utc=True)\n        frame["checkpoint_utc"] = pd.to_datetime(frame["checkpoint_utc"], utc=True)\n    return train_df, val_df\n\n\ndef build_expanding_folds(\n    train_df: pd.DataFrame,\n    val_df: pd.DataFrame,\n    observability_buffer_hours: int = 12,\n) -> list[tuple[int, pd.DataFrame, pd.DataFrame]]:\n    """Create the 4 expanding chronological folds.\n\n    All rows in train_T2h.csv are already verified observable before Sep 1.\n    Earlier validation rows added to later folds are filtered conservatively so their\n    scheduled departure is at least `observability_buffer_hours` before the fold\'s\n    first checkpoint_utc (unless Person 1\'s explicit fold/observability column is used).\n    """\n    folds = []\n    for fold_id, start_date, end_date in FOLDS:\n        val_mask = val_df["FlightDate"].between(start_date, end_date)\n        val_fold = val_df.loc[val_mask].copy()\n        if val_fold.empty:\n            raise ValueError(f"Fold {fold_id} ({start_date}..{end_date}) has 0 validation rows.")\n\n        first_ckpt = val_fold["checkpoint_utc"].min()\n        cutoff = first_ckpt - pd.Timedelta(hours=observability_buffer_hours)\n\n        earlier_val_mask = (val_df["FlightDate"] < start_date) & (val_df["scheduled_dep_utc"] <= cutoff)\n        train_fold = pd.concat([train_df, val_df.loc[earlier_val_mask]], ignore_index=True)\n        train_fold = train_fold.sort_values(["checkpoint_utc", "row_id"]).reset_index(drop=True)\n        val_fold = val_fold.sort_values(["checkpoint_utc", "row_id"]).reset_index(drop=True)\n        folds.append((fold_id, train_fold, val_fold))\n    return folds\n\n\nclass FoldPreprocessor(TransformerMixin, BaseEstimator):\n    """Fold-safe preprocessor matching the T-2h feature guide (~80 encoded features).\n\n    - Learns top-20 Dest categories in the training fold (others -> \'Other\')\n    - Preserves missing/unknown categorical values as \'__MISSING__\'\n    - Adds 4 cyclic sin/cos features for dep_hour and month\n    - Median-imputes numeric features and standard-scales all features inside the fold\n    """\n\n    def __init__(self, top_k_dest: int = 20):\n        self.top_k_dest = top_k_dest\n        self.top_dests_: set[str] = set()\n        self.encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)\n        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)\n        self.scaler = StandardScaler()\n        self.num_cols_: list[str] = []\n        self.feature_names_: list[str] = []\n\n    def _prepare_frame(self, df: pd.DataFrame, fit: bool) -> tuple[pd.DataFrame, pd.DataFrame]:\n        feat_cols = get_feature_columns(df)\n        work = df[feat_cols].copy()\n\n        # Cyclic features (4 extra columns)\n        work["dep_hour_sin"] = np.sin(2 * np.pi * work["dep_hour"].astype(float) / 24.0)\n        work["dep_hour_cos"] = np.cos(2 * np.pi * work["dep_hour"].astype(float) / 24.0)\n        work["month_sin"] = np.sin(2 * np.pi * (work["month"].astype(float) - 1.0) / 12.0)\n        work["month_cos"] = np.cos(2 * np.pi * (work["month"].astype(float) - 1.0) / 12.0)\n\n        # Categorical preparation\n        cat_df = pd.DataFrame(index=work.index)\n        for col in [name for name in CAT_COLS if name in work]:\n            s = work[col].astype("string").fillna("__MISSING__")\n            if col == "Dest":\n                if fit:\n                    self.top_dests_ = set(s.value_counts().head(self.top_k_dest).index.tolist())\n                s = s.where(s.isin(self.top_dests_), "Other")\n            cat_df[col] = s\n\n        num_cols = [c for c in work.columns if c not in CAT_COLS]\n        if fit:\n            self.num_cols_ = num_cols\n        num_df = work[self.num_cols_].astype(np.float32)\n        return cat_df, num_df\n\n    def fit(self, df: pd.DataFrame, y=None):\n        cat_df, num_df = self._prepare_frame(df, fit=True)\n        self.imputer.fit(num_df)\n        self.feature_names_ = self.num_cols_.copy()\n        if cat_df.shape[1]:\n            self.encoder.fit(cat_df)\n            self.feature_names_ += list(self.encoder.get_feature_names_out())\n        self.scaler.fit(self._matrix(cat_df, num_df))\n        return self\n\n    def _matrix(self, cats, nums):\n        values = self.imputer.transform(nums)\n        if cats.shape[1]:\n            values = np.hstack([values, self.encoder.transform(cats)])\n        return values.astype(np.float32)\n\n    def transform(self, df: pd.DataFrame) -> np.ndarray:\n        return self.scaler.transform(self._matrix(*self._prepare_frame(df, fit=False)))\n\n\ndef get_tuning_slice(X: np.ndarray, y: np.ndarray, max_rows: int) -> tuple[np.ndarray, np.ndarray]:\n    """Return a continuous chronological block (most recent `max_rows`) for fast tuning."""\n    if max_rows <= 0 or len(X) <= max_rows:\n        return X, y\n    return X[-max_rows:], y[-max_rows:]\n\n\ndef score_probs(y_true: np.ndarray, probs: np.ndarray) -> dict[str, float]:\n    return {\n        "ap": float(average_precision_score(y_true, probs)),\n        "roc_auc": float(roc_auc_score(y_true, probs)),\n    }\n\n\ndef tune_on_block(\n    model_name: str,\n    X_tune: np.ndarray,\n    y_tune: np.ndarray,\n    seed: int = 11,\n) -> dict[str, Any]:\n    """Internal chronological 80/20 split on the continuous tuning block."""\n    split = int(len(X_tune) * 0.8)\n    X_tr, y_tr = X_tune[:split], y_tune[:split]\n    X_va, y_va = X_tune[split:], y_tune[split:]\n\n    if model_name == "decision_tree":\n        grid = [\n            {"max_depth": d, "min_samples_leaf": leaf, "ccp_alpha": alpha}\n            for d in (6, 8, 10, 12)\n            for leaf in (100, 300)\n            for alpha in (0.0, 1e-5, 1e-4)\n        ]\n        builder = lambda p: DecisionTreeClassifier(random_state=seed, **p)\n    elif model_name == "random_forest":\n        grid = [\n            {"n_estimators": 100, "max_depth": d, "min_samples_leaf": leaf, "n_jobs": -1}\n            for d in (10, 14, 18)\n            for leaf in (50, 200)\n        ]\n        builder = lambda p: RandomForestClassifier(random_state=seed, **p)\n    elif model_name == "adaboost":\n        grid = [\n            {"n_estimators": n, "learning_rate": lr, "max_depth": d}\n            for n in (50, 100)\n            for lr in (0.1, 0.5, 1.0)\n            for d in (1, 2)\n        ]\n        builder = lambda p: AdaBoostClassifier(\n            estimator=DecisionTreeClassifier(max_depth=p["max_depth"], random_state=seed),\n            n_estimators=p["n_estimators"],\n            learning_rate=p["learning_rate"],\n            random_state=seed,\n        )\n    else:\n        raise ValueError(f"Unknown model_name: {model_name}")\n\n    best_ap = -1.0\n    best_params = grid[0]\n    for params in grid:\n        clf = builder(params)\n        clf.fit(X_tr, y_tr)\n        probs = clf.predict_proba(X_va)[:, 1]\n        ap = average_precision_score(y_va, probs)\n        if ap > best_ap:\n            best_ap = ap\n            best_params = params\n    return best_params\n\n\ndef build_estimator(model_name: str, params: dict[str, Any], seed: int):\n    if model_name == "constant":\n        return DummyClassifier(strategy="prior")\n    if model_name == "decision_tree":\n        return DecisionTreeClassifier(random_state=seed, **params)\n    if model_name == "random_forest":\n        return RandomForestClassifier(random_state=seed, **params)\n    if model_name == "adaboost":\n        return AdaBoostClassifier(\n            estimator=DecisionTreeClassifier(max_depth=params["max_depth"], random_state=seed),\n            n_estimators=params["n_estimators"],\n            learning_rate=params["learning_rate"],\n            random_state=seed,\n        )\n    if model_name == "logreg":\n        return LogisticRegression(max_iter=500, C=params.get("C", 1.0), random_state=seed)\n    raise ValueError(f"Unknown model_name: {model_name}")\n\n\ndef run_pipeline(\n    data_dir: Path,\n    out_dir: Path,\n    models_to_run: list[str],\n    tune_rows: int = 100_000,\n    pca_variance: float = 0.95,\n    max_train_rows: int = 0,\n) -> None:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    t0 = time.time()\n    train_df, val_df = load_dev_data(data_dir)\n    folds = build_expanding_folds(train_df, val_df)\n\n    score_rows = []\n    pca_rows = []\n    chosen_settings: dict[str, Any] = {\n        "seeds": list(SEEDS),\n        "tune_rows": tune_rows,\n        "pca_variance_threshold": pca_variance,\n        "folds": FOLDS,\n        "models": {},\n    }\n\n    oof_frames: dict[int, pd.DataFrame] = {}\n    for fold_id, _, val_fold in folds:\n        oof_frames[fold_id] = val_fold[["row_id", "FlightDate", "LATE"]].copy()\n        oof_frames[fold_id]["fold"] = fold_id\n\n    for fold_id, train_fold, val_fold in folds:\n        f_t0 = time.time()\n        if max_train_rows > 0 and len(train_fold) > max_train_rows:\n            train_fold = train_fold.iloc[-max_train_rows:].reset_index(drop=True)\n\n        prep = FoldPreprocessor(top_k_dest=20)\n        X_train = prep.fit_transform(train_fold)\n        y_train = train_fold[TARGET_COL].to_numpy(dtype=int)\n        X_val = prep.transform(val_fold)\n        y_val = val_fold[TARGET_COL].to_numpy(dtype=int)\n\n        X_tune, y_tune = get_tuning_slice(X_train, y_train, tune_rows)\n        print(\n            f"\\n[Fold {fold_id}] train={len(X_train):,} val={len(X_val):,} "\n            f"features={X_train.shape[1]} pos_rate(val)={y_val.mean():.4f}"\n        )\n\n        # Tune once on Fold 1 (or per fold; here we tune on Fold 1 and lock, or tune per fold)\n        for model_name in models_to_run:\n            if model_name == "pca_logreg":\n                # Compare the exact same classifier (LogisticRegression) without vs with PCA\n                pca = PCA(n_components=pca_variance, svd_solver="full", random_state=SEEDS[0])\n                X_train_pca = pca.fit_transform(X_train)\n                X_val_pca = pca.transform(X_val)\n                n_comp = int(pca.n_components_)\n                exp_var = float(pca.explained_variance_ratio_.sum())\n\n                for variant, X_tr_v, X_va_v in [\n                    ("logreg_no_pca", X_train, X_val),\n                    ("logreg_with_pca", X_train_pca, X_val_pca),\n                ]:\n                    seed_probs = []\n                    seed_aps = []\n                    seed_aucs = []\n                    for seed in SEEDS:\n                        clf = build_estimator("logreg", {"C": 1.0}, seed)\n                        clf.fit(X_tr_v, y_train)\n                        probs = clf.predict_proba(X_va_v)[:, 1]\n                        m = score_probs(y_val, probs)\n                        seed_probs.append(probs)\n                        seed_aps.append(m["ap"])\n                        seed_aucs.append(m["roc_auc"])\n                    mean_probs = np.mean(seed_probs, axis=0)\n                    oof_frames[fold_id][f"prob_{variant}"] = mean_probs\n                    score_rows.append(\n                        {\n                            "model": variant,\n                            "fold": fold_id,\n                            "ap": float(np.mean(seed_aps)),\n                            "roc_auc": float(np.mean(seed_aucs)),\n                            "n_features": X_tr_v.shape[1],\n                        }\n                    )\n                pca_rows.append(\n                    {\n                        "fold": fold_id,\n                        "original_features": X_train.shape[1],\n                        "retained_components": n_comp,\n                        "explained_variance_ratio_sum": exp_var,\n                    }\n                )\n                chosen_settings["models"]["pca_logreg"] = {\n                    "classifier": "LogisticRegression(max_iter=500, C=1.0)",\n                    "pca_variance_threshold": pca_variance,\n                }\n                print(\n                    f"  pca_logreg: retained {n_comp}/{X_train.shape[1]} components "\n                    f"(explained_var={exp_var:.4f})"\n                )\n                continue\n\n            if model_name == "constant":\n                params: dict[str, Any] = {}\n            else:\n                if model_name not in chosen_settings["models"]:\n                    params = tune_on_block(model_name, X_tune, y_tune, seed=SEEDS[0])\n                    chosen_settings["models"][model_name] = params\n                    print(f"  {model_name} tuned params: {params}")\n                else:\n                    params = chosen_settings["models"][model_name]\n\n            seed_probs = []\n            seed_aps = []\n            seed_aucs = []\n            for seed in SEEDS:\n                clf = build_estimator(model_name, params, seed)\n                clf.fit(X_train, y_train)\n                probs = clf.predict_proba(X_val)[:, 1]\n                m = score_probs(y_val, probs)\n                seed_probs.append(probs)\n                seed_aps.append(m["ap"])\n                seed_aucs.append(m["roc_auc"])\n\n            mean_probs = np.mean(seed_probs, axis=0)\n            oof_frames[fold_id][f"prob_{model_name}"] = mean_probs\n            ap_fold = float(np.mean(seed_aps))\n            auc_fold = float(np.mean(seed_aucs))\n            score_rows.append(\n                {\n                    "model": model_name,\n                    "fold": fold_id,\n                    "ap": ap_fold,\n                    "roc_auc": auc_fold,\n                    "n_features": X_train.shape[1],\n                }\n            )\n            print(f"  {model_name}: AP={ap_fold:.5f}, ROC-AUC={auc_fold:.5f}")\n\n        print(f"  Fold {fold_id} finished in {time.time() - f_t0:.1f}s")\n\n    scores_df = pd.DataFrame(score_rows)\n    summary_df = (\n        scores_df.groupby("model", as_index=False)\n        .agg(\n            ap_mean=("ap", "mean"),\n            ap_std=("ap", "std"),\n            roc_auc_mean=("roc_auc", "mean"),\n            roc_auc_std=("roc_auc", "std"),\n        )\n        .sort_values("ap_mean", ascending=False)\n    )\n\n    scores_df.to_csv(out_dir / "person2_fold_scores.csv", index=False)\n    summary_df.to_csv(out_dir / "person2_summary_scores.csv", index=False)\n    if pca_rows:\n        pd.DataFrame(pca_rows).to_csv(out_dir / "person2_pca_stats.csv", index=False)\n\n    oof_all = pd.concat([oof_frames[fid] for fid, _, _ in FOLDS], ignore_index=True)\n    try:\n        oof_all.to_parquet(out_dir / "person2_oof_predictions.parquet", index=False)\n    except ImportError:\n        oof_all.to_csv(out_dir / "person2_oof_predictions.csv", index=False)\n\n    chosen_settings["command"] = " ".join(sys.argv)\n    chosen_settings["elapsed_seconds"] = round(time.time() - t0, 2)\n    with (out_dir / "person2_chosen_settings.json").open("w", encoding="utf-8") as f:\n        json.dump(chosen_settings, f, indent=2)\n\n    print("\\n=== Person 2 Summary (mean +/- std across 4 folds) ===")\n    print(summary_df.to_string(index=False))\n    print(f"\\nSaved Person 4 handoff files to: {out_dir.resolve()}")\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Person 2 T-2h Flight Delay Pipeline")\n    parser.add_argument("--data-dir", type=Path, default=Path("final_csv"), help="Folder containing train_T2h.csv and validation_T2h.csv")\n    parser.add_argument("--out-dir", type=Path, default=Path("person2_outputs"), help="Output folder for Person 4 handoff files")\n    parser.add_argument(\n        "--models",\n        nargs="+",\n        default=["constant", "decision_tree", "random_forest", "adaboost", "pca_logreg"],\n        choices=["constant", "decision_tree", "random_forest", "adaboost", "pca_logreg"],\n        help="Models to run",\n    )\n    parser.add_argument("--tune-rows", type=int, default=100_000, help="Continuous block size inside training fold for tuning (0 = full fold)")\n    parser.add_argument("--pca-variance", type=float, default=0.95, help="Variance threshold for PCA")\n    parser.add_argument("--max-train-rows", type=int, default=0, help="Optional cap on training rows per fold (0 = all)")\n    return parser.parse_args()\n\n\nif __name__ == "__main__":\n    args = parse_args()\n    run_pipeline(\n        data_dir=args.data_dir,\n        out_dir=args.out_dir,\n        models_to_run=args.models,\n        tune_rows=args.tune_rows,\n        pca_variance=args.pca_variance,\n        max_train_rows=args.max_train_rows,\n    )\n', encoding='utf-8')
Path('train.py').write_text('"""T-2h chronological development training. See train.md for commands."""\n\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.decomposition import PCA\nfrom sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OrdinalEncoder\nfrom threadpoolctl import threadpool_limits\n\nfrom build_features import HISTORY, META, ROOT, SHARED, WX\nfrom person2.person2_pipeline import (\n    CAT_COLS,\n    FOLDS,\n    SEEDS,\n    FoldPreprocessor,\n    build_estimator,\n)\n\nSUFFIX = "_T2h"\nWEATHER = ["wx_" + x + SUFFIX for x in WX] + ["codes_known_T2h", "wx_unmatched_T2h"]\nACTIVITY = [x + SUFFIX for x in HISTORY]\nALL = SHARED + WEATHER + ACTIVITY\nLADDER = {\n    "schedule": SHARED[:7],\n    "route": SHARED,\n    "weather": SHARED + WEATHER,\n    "full": ALL,\n}\nDEFAULTS = {\n    "constant": {},\n    "decision_tree": {"max_depth": 12, "min_samples_leaf": 50, "ccp_alpha": 0.00001},\n    "random_forest": {"n_estimators": 100, "max_depth": 16, "min_samples_leaf": 12},\n    "logreg": {"C": 1.0},\n    "pca_logreg": {"C": 1.0},\n    "adaboost": {"max_depth": 2, "n_estimators": 60, "learning_rate": 0.1},\n    "gradient": {"n_estimators": 80, "max_depth": 3, "learning_rate": 0.08},\n    "histgb": {"max_iter": 120, "max_leaf_nodes": 31},\n    "xgboost": {"n_estimators": 150, "max_depth": 6, "subsample": 0.85},\n    "lightgbm": {\n        "n_estimators": 150,\n        "num_leaves": 31,\n        "subsample": 0.85,\n        "subsample_freq": 1,\n    },\n    "catboost": {"iterations": 150, "depth": 6},\n}\n\n\nclass NativeCategories(TransformerMixin, BaseEstimator):\n    """Fold-specific pandas categories or explicit missing strings for CatBoost."""\n\n    def __init__(self, cats, as_strings=False):\n        self.cats = cats\n        self.as_strings = as_strings\n\n    def fit(self, X, y=None):\n        self.levels_ = {k: sorted(X[k].dropna().unique()) for k in self.cats}\n        return self\n\n    def transform(self, X):\n        frame = X.copy()\n        for col, levels in self.levels_.items():\n            if self.as_strings:\n                frame[col] = (\n                    frame[col].astype("string").fillna("__MISSING__").astype(object)\n                )\n            else:\n                frame[col] = pd.Categorical(frame[col], categories=levels)\n        return frame\n\n\ndef model_pipeline(name, cols, seed, jobs, overrides):\n    params = DEFAULTS[name] | overrides\n    cats = [c for c in CAT_COLS if c in cols]\n    steps = []\n    if name == "histgb":\n        preprocess = ColumnTransformer(\n            [\n                (\n                    "categorical",\n                    OrdinalEncoder(\n                        max_categories=254,\n                        handle_unknown="use_encoded_value",\n                        unknown_value=np.nan,\n                    ),\n                    cats,\n                )\n            ],\n            remainder="passthrough",\n        )\n        model = HistGradientBoostingClassifier(\n            **params,\n            categorical_features=list(range(len(cats))) or None,\n            random_state=seed,\n            early_stopping=False,\n        )\n    elif name in ("xgboost", "lightgbm", "catboost"):\n        preprocess = NativeCategories(cats, as_strings=name == "catboost")\n        if name == "xgboost":\n            from xgboost import XGBClassifier\n\n            model = XGBClassifier(\n                **params,\n                tree_method="hist",\n                enable_categorical=True,\n                random_state=seed,\n                n_jobs=jobs,\n            )\n        elif name == "lightgbm":\n            from lightgbm import LGBMClassifier\n\n            model = LGBMClassifier(\n                **params, random_state=seed, n_jobs=jobs, verbosity=-1\n            )\n        else:\n            from catboost import CatBoostClassifier\n\n            model = CatBoostClassifier(\n                **params,\n                cat_features=cats,\n                random_seed=seed,\n                thread_count=jobs,\n                verbose=False,\n                allow_writing_files=False,\n                has_time=True,\n            )\n    else:\n        preprocess = FoldPreprocessor()\n        if name == "gradient":\n            model = GradientBoostingClassifier(\n                **params, random_state=seed, n_iter_no_change=None\n            )\n        else:\n            if name == "random_forest":\n                params["n_jobs"] = jobs\n            model = build_estimator(\n                "logreg" if name == "pca_logreg" else name, params, seed\n            )\n        if name == "pca_logreg":\n            steps.append(("pca", PCA(n_components=0.95, svd_solver="full")))\n    return Pipeline([("preprocess", preprocess), *steps, ("classifier", model)])\n\n\ndef read_dev(directory):\n    data = []\n    for part, start, end in [\n        ("train", "2025-01-01", "2025-08-30"),\n        ("validation", FOLDS[0][1], FOLDS[-1][2]),\n    ]:\n        frame = pd.read_csv(directory / f"{part}_T2h.csv", parse_dates=META[2:])\n        if (\n            list(frame) != META + ALL + ["LATE"]\n            or not frame.FlightDate.between(start, end).all()\n        ):\n            raise ValueError(\n                f"Unexpected {part} dates or schema: {frame.columns.tolist()}"\n            )\n        data.append(frame)\n    dev = (\n        pd.concat(data, ignore_index=True)\n        .set_index("row_id", verify_integrity=True)\n        .sort_values("checkpoint_utc", kind="stable")\n    )\n    assert dev.LATE.isin([0, 1]).all()\n    assert (dev.scheduled_dep_utc - dev.checkpoint_utc).eq(pd.Timedelta(hours=2)).all()\n    age = (dev.checkpoint_utc - dev.wx_obs_time).dt.total_seconds() / 60\n    np.testing.assert_allclose(age, dev.wx_obs_age_min_T2h, equal_nan=True)\n    assert (\n        age.dropna().between(0, 180).all() and age.isna().eq(dev.wx_unmatched_T2h).all()\n    )\n    assert (\n        dev[["wx_" + k + SUFFIX for k in ["ts", "fg", "ra", "sn", "fz", "br_hz"]]]\n        .notna()\n        .eq(dev.codes_known_T2h, axis=0)\n        .all()\n        .all()\n    )\n    return dev\n\n\ndef raw_events(path, dev):\n    # The dataset builder concatenates whole BTS months in order. Predicate pushdown\n    # preserves original row positions for this pre-December prefix, including warm-up.\n    cols = [\n        "Origin",\n        "FlightDate",\n        "scheduled_dep_utc",\n        "Cancelled",\n        "DepDelay",\n        "DepDel15",\n    ]\n    events = pd.read_parquet(\n        path, columns=cols, filters=[("FlightDate", "<", "2025-12-01")]\n    ).reset_index(drop=True)\n    matched = events.loc[dev.index]\n    for col in ["Origin", "FlightDate", "scheduled_dep_utc"]:\n        np.testing.assert_array_equal(matched[col], dev[col])\n    np.testing.assert_array_equal(matched.DepDel15, dev.LATE)\n    events["actual"] = events.scheduled_dep_utc + pd.to_timedelta(\n        events.DepDelay, unit="min"\n    )\n    return events\n\n\ndef history_check(dev, events):\n    eligible = (\n        events.Cancelled.eq(0) & events.DepDelay.notna() & events.DepDel15.notna()\n    )\n    blocked = events.Origin.eq("ATL") & (\n        events.FlightDate.between("2025-01-27", "2025-01-31")\n        | events.FlightDate.between("2025-05-13", "2025-05-29")\n    )\n    checks = []\n    for idx, r in (\n        dev[dev.FlightDate.ge("2025-09-01")].sample(5, random_state=42).iterrows()\n    ):\n        t = r.checkpoint_utc\n        same = events.Origin.eq(r.Origin)\n        past = events.loc[\n            same\n            & eligible\n            & ~blocked\n            & events.actual.gt(t - pd.Timedelta(hours=1))\n            & events.actual.le(t)\n        ]\n        past = past.drop(idx, errors="ignore")\n        demand = (\n            same\n            & events.scheduled_dep_utc.gt(t)\n            & events.scheduled_dep_utc.le(t + pd.Timedelta(hours=1))\n        ).sum()\n        expected = [\n            len(past),\n            past.DepDelay.mean(),\n            past.DepDelay.ge(15).mean(),\n            demand,\n            int(bool(len(past))),\n        ]\n        np.testing.assert_allclose(r[ACTIVITY].astype(float), expected, equal_nan=True)\n        checks.append(\n            dict(\n                row_id=idx,\n                origin=r.Origin,\n                checkpoint=t,\n                included_ids=" ".join(map(str, past.index)),\n                **dict(zip(ACTIVITY, expected)),\n            )\n        )\n    return pd.DataFrame(checks)\n\n\ndef folds(dev, actual, smoke):\n    if smoke:\n        block = dev.iloc[:20000]\n        first = block.checkpoint_utc.iloc[16000]\n        yield (\n            0,\n            block[(block.checkpoint_utc < first) & (actual.loc[block.index] < first)],\n            block[block.checkpoint_utc >= first],\n        )\n        return\n    for fold, start, end in FOLDS:\n        val = dev[dev.FlightDate.between(start, end)]\n        t = val.checkpoint_utc.min()\n        train = dev[(dev.FlightDate < start) & (dev.checkpoint_utc < t) & (actual < t)]\n        assert train.checkpoint_utc.max() < t and actual.loc[train.index].max() < t\n        yield fold, train, val\n\n\ndef column_audit():\n    rows = []\n    for line in (ROOT / "COLUMN_AUDIT.md").read_text().splitlines():\n        if line.startswith("| ") and not line.startswith("| Input"):\n            name, kind, reason = [x.strip() for x in line.strip("|").split("|")]\n            rows.append(\n                (name if name in SHARED else name + SUFFIX, kind, reason, "predictor")\n            )\n    assert {r[0] for r in rows} == set(ALL)\n    rows.extend(\n        (c, "audit", "Identifier or timestamp used only for verification.", "metadata")\n        for c in META\n    )\n    rows.append(\n        ("LATE", "outcome", "Departure delay of at least 15 minutes.", "target")\n    )\n    return pd.DataFrame(rows, columns=["column", "type", "reason", "role"])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\n        "--models", nargs="+", choices=list(DEFAULTS), default=["constant", "histgb"]\n    )\n    parser.add_argument("--stage", choices=LADDER, default="full")\n    parser.add_argument("--drop", nargs="+", choices=ACTIVITY, default=[])\n    parser.add_argument("--seeds", nargs="+", type=int, default=list(SEEDS))\n    parser.add_argument("--smoke", action="store_true")\n    parser.add_argument(\n        "--refit",\n        action="store_true",\n        help="Refit one locked configuration on all development rows and save its pipeline.",\n    )\n    parser.add_argument("--jobs", type=int, default=4)\n    parser.add_argument(\n        "--params",\n        default="{}",\n        help="JSON object of hyperparameter overrides for one estimator",\n    )\n    parser.add_argument(\n        "--train-limit",\n        type=int,\n        default=0,\n        help="Latest N eligible training rows per fold; 0 uses all",\n    )\n    parser.add_argument("--out", type=Path, required=True)\n    parser.add_argument("--data-dir", type=Path, default=ROOT / "final_csv")\n    parser.add_argument(\n        "--events", type=Path, default=ROOT / "fresh_2025/output_with_weather.parquet"\n    )\n    args = parser.parse_args()\n    overrides = json.loads(args.params)\n    if not isinstance(overrides, dict) or (\n        overrides\n        and (\n            len(args.models) != 1\n            or not overrides.keys() <= DEFAULTS[args.models[0]].keys()\n        )\n    ):\n        parser.error(\n            "Use one model and only its documented configuration keys for overrides."\n        )\n    if (\n        args.jobs < 1\n        or args.train_limit < 0\n        or (args.out.exists() and any(args.out.iterdir()))\n    ):\n        parser.error(\n            "Choose positive jobs, nonnegative train-limit, and an empty output directory."\n        )\n    if args.refit and (\n        len(args.models) != 1 or len(args.seeds) != 1 or args.smoke or args.train_limit\n    ):\n        parser.error(\n            "Refit requires exactly one model, one seed, and all development rows."\n        )\n    if len(set(args.models)) != len(args.models) or len(set(args.seeds)) != len(\n        args.seeds\n    ):\n        parser.error("Models and seeds must be unique within a run.")\n    cols = [c for c in LADDER[args.stage] if c not in args.drop]\n    dev = read_dev(args.data_dir)\n    events = raw_events(args.events, dev)\n    actual = events.actual.loc[dev.index]\n    checks = history_check(dev, events)\n    args.out.mkdir(parents=True, exist_ok=True)\n    checks.to_csv(args.out / "history_checks.csv", index=False)\n    audit = column_audit()\n    audit["selected"] = audit.column.isin(cols)\n    audit.to_csv(args.out / "column_audit.csv", index=False)\n    print("Development schema:", META + ALL + ["LATE"], flush=True)\n    print(\n        "Development weather codes known:",\n        dev.codes_known_T2h.mean(),\n        "unmatched:",\n        dev.wx_unmatched_T2h.sum(),\n        flush=True,\n    )\n    print(checks.drop(columns="included_ids").to_string(index=False), flush=True)\n    (args.out / "config.json").write_text(\n        json.dumps(\n            {"arguments": vars(args), "features": cols, "defaults": DEFAULTS},\n            indent=2,\n            default=str,\n        )\n    )\n    if args.refit:\n        # Classes in a saved pipeline must resolve by importable module name.\n        from train import model_pipeline as saved_pipeline\n\n        model = saved_pipeline(\n            args.models[0], cols, args.seeds[0], args.jobs, overrides\n        )\n        start = time.perf_counter()\n        with threadpool_limits(limits=args.jobs):\n            model.fit(dev[cols], dev.LATE.astype(int))\n        elapsed = time.perf_counter() - start\n        joblib.dump(model, args.out / "model.joblib", compress=3)\n        print(f"Refitted on {len(dev)} development rows in {elapsed:.1f}s.")\n        return\n    scores = []\n    for fold, train, val in folds(dev, actual, args.smoke):\n        if args.train_limit:\n            train = train.tail(args.train_limit)\n        print(\n            f"Fold {fold} train={len(train)} late={train.LATE.mean():.4f} validation={len(val)} late={val.LATE.mean():.4f}",\n            flush=True,\n        )\n        for name in args.models:\n            for seed in args.seeds:\n                pipe = model_pipeline(name, cols, seed, args.jobs, overrides)\n                start = time.perf_counter()\n                with threadpool_limits(limits=args.jobs):\n                    pipe.fit(train[cols], train.LATE.astype(int))\n                    prob = pipe.predict_proba(val[cols])[:, 1]\n                seconds = time.perf_counter() - start\n                result = {\n                    "model": name,\n                    "stage": args.stage,\n                    "fold": fold,\n                    "seed": seed,\n                    "train_rows": len(train),\n                    "validation_rows": len(val),\n                    "train_late": train.LATE.mean(),\n                    "validation_late": val.LATE.mean(),\n                    "ap": average_precision_score(val.LATE, prob),\n                    "roc_auc": roc_auc_score(val.LATE, prob),\n                    "seconds": seconds,\n                }\n                if "pca" in pipe.named_steps:\n                    result.update(\n                        components=pipe["pca"].n_components_,\n                        explained_variance=pipe["pca"].explained_variance_ratio_.sum(),\n                    )\n                scores.append(result)\n                val[["FlightDate", "LATE"]].assign(probability=prob).to_parquet(\n                    args.out / f"pred_{name}_{fold}_{seed}.parquet"\n                )\n                pd.DataFrame(scores).to_csv(args.out / "fold_scores.csv", index=False)\n                print(\n                    name,\n                    seed,\n                    f"AP={result[\'ap\']:.4f} ROC-AUC={result[\'roc_auc\']:.4f} {seconds:.1f}s",\n                    flush=True,\n                )\n    per_fold = pd.DataFrame(scores).groupby(["model", "fold"])[["ap", "roc_auc"]].mean()\n    summary = per_fold.groupby("model").agg(["mean", "std"])\n    summary.columns = ["_".join(c) for c in summary.columns]\n    summary.to_csv(args.out / "results.csv")\n    print(summary)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('COLUMN_AUDIT.md').write_text('# Input column audit\n\nEach horizon uses the following 35 raw inputs. Append `_T2h` or `_T6h` to all weather and activity names. Weather measurements and their flags use the latest same-origin report at or before the checkpoint, up to 180 minutes old.\n\n| Input | Type | Reason or meaning |\n|---|---|---|\n| month | Known in advance | Scheduled local flight month |\n| dow | Known in advance | Scheduled local weekday, Monday = 0 |\n| dep_hour | Known in advance | Scheduled origin-local departure hour |\n| arr_hour | Known in advance | Scheduled destination-local arrival hour |\n| CRSElapsedTime | Known in advance | Scheduled flight duration in minutes |\n| Distance | Static | Published route distance in miles |\n| is_holiday | Known in advance | Observed US federal holiday on the local flight date |\n| Origin | Static | Departure airport |\n| Dest | Known in advance | Scheduled destination |\n| Reporting_Airline | Known in advance | Reporting carrier on the schedule |\n| wx_skyc1 | Observed by T | First reported sky-cover layer |\n| wx_tmpf | Observed by T | Temperature in °F |\n| wx_dwpf | Observed by T | Dew point in °F |\n| wx_drct | Observed by T | Wind direction in degrees true |\n| wx_sknt | Observed by T | Wind speed in knots |\n| wx_gust | Observed by T | Gust speed in knots |\n| wx_vsby | Observed by T | Visibility in statute miles |\n| wx_p01i | Observed by T | Reported precipitation in inches, trace = 0.0001; one report per prediction |\n| wx_skyl1 | Observed by T | First cloud-layer base in feet |\n| wx_obs_age_min | Observed by T | Minutes from report observation time to checkpoint |\n| wx_ts | Observed by T | Thunderstorm in reported codes |\n| wx_fg | Observed by T | Fog in reported codes |\n| wx_ra | Observed by T | Rain in reported codes |\n| wx_sn | Observed by T | Snow in reported codes |\n| wx_fz | Observed by T | Freezing descriptor in reported codes |\n| wx_br_hz | Observed by T | Mist or haze in reported codes |\n| codes_known | Observed by T | Report contains a weather-code value; absent codes leave event flags missing |\n| wx_gust_missing | Observed by T | Matched report lacks a gust measurement |\n| wx_skyl1_missing | Observed by T | Matched report lacks a first-layer height |\n| wx_unmatched | Observed by T | No valid same-origin report within three hours |\n| n_departed_60min | Observed by T | Number of other same-origin departures in (T−60 min, T] |\n| prior_mean_delay_60min | Observed by T | Mean signed departure delay in minutes among those departures |\n| late_rate_60min | Observed by T | Fraction of those departures delayed at least 15 minutes |\n| has_history | Observed by T | One when the eligible historical departure count is positive |\n| n_scheduled_next_60min | Known in advance | Same-origin scheduled departures in (T, T+60 min], irrespective of actual operation |\n\nExclude five metadata columns from X: row_id (cross-horizon record identifier), FlightDate (split date), scheduled_dep_utc, checkpoint_utc and wx_obs_time (timestamp audit). LATE is the outcome. Current-flight realized times, delay causes, tail and flight numbers are excluded from all model matrices. For ablations group cloud coverage with weather.\n', encoding='utf-8')
print('Ready: train.py, person2/person2_pipeline.py, COLUMN_AUDIT.md')

!pip -q install scikit-learn pandas numpy pyarrow airportsdata requests joblib threadpoolctl


In [ ]:
# 2. Build train_T2h.csv, validation_T2h.csv, and fresh_2025/output_with_weather.parquet if missing
from pathlib import Path

DATA_DIR = Path('final_csv')
EVENTS = Path('fresh_2025/output_with_weather.parquet')

if not (DATA_DIR / 'train_T2h.csv').exists() or not (DATA_DIR / 'validation_T2h.csv').exists() or not EVENTS.exists():
    print('Building dataset from BTS + IEM (this takes a few minutes)...')
    Path('pipeline').mkdir(exist_ok=True)
    Path('pipeline/__init__.py').write_text('', encoding='utf-8')
    Path('pipeline/bts.py').write_text('"""Build a 2025 departure cohort and a separate history of flights into its airports."""\n\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor\nfrom datetime import UTC, date, datetime, timedelta\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport pandas as pd\nimport requests\n\nYEAR = 2025\nAIRPORTS = {"ATL", "ORD", "LAX"}\nROOT = Path(__file__).resolve().parents[1] / "fresh_2025"\nBASE = (\n    "https://transtats.bts.gov/PREZIP/"\n    "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"\n)\nFIELDS = [\n    "FlightDate",\n    "Reporting_Airline",\n    "Tail_Number",\n    "Flight_Number_Reporting_Airline",\n    "Origin",\n    "Dest",\n    "CRSDepTime",\n    "DepTime",\n    "DepDelay",\n    "DepDel15",\n    "CRSArrTime",\n    "ArrTime",\n    "ArrDelay",\n    "Cancelled",\n    "Diverted",\n    "CRSElapsedTime",\n    "Distance",\n]\n\n\ndef download_month(year, month):\n    url = BASE.format(year=year, month=month)\n    path = ROOT / "bts_archives" / f"{year}-{month:02}.zip"\n    path.parent.mkdir(exist_ok=True)\n    if path.exists():\n        return path\n    partial = path.with_suffix(".part")\n    for attempt in range(3):\n        try:\n            with requests.get(url, stream=True, timeout=(20, 180)) as response:\n                response.raise_for_status()\n                with partial.open("wb") as out:\n                    for chunk in response.iter_content(1024 * 1024):\n                        out.write(chunk)\n            with zipfile.ZipFile(partial) as archive:\n                if archive.testzip() is not None:\n                    raise zipfile.BadZipFile("CRC failure")\n            partial.replace(path)\n            with (ROOT / "bts_download_log.txt").open("a") as log:\n                print(\n                    datetime.now(UTC).isoformat(),\n                    url,\n                    path.stat().st_size,\n                    file=log,\n                )\n            return path\n        except (requests.RequestException, zipfile.BadZipFile):\n            if attempt == 2:\n                raise\n            time.sleep(5 * 2**attempt)\n\n\ndef main():\n    ROOT.mkdir(exist_ok=True)\n    targets, inbound = [], []\n    # Two lookback days also cover US local-date differences for a 24-hour history.\n    earliest = (date(YEAR, 1, 1) - timedelta(days=2)).isoformat()\n    months = [(YEAR - 1, 12)] + [(YEAR, m) for m in range(1, 13)]\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        jobs = [pool.submit(download_month, year, month) for year, month in months]\n        for (year, month), job in zip(months, jobs):\n            with zipfile.ZipFile(job.result()) as archive:\n                (name,) = [n for n in archive.namelist() if n.lower().endswith(".csv")]\n                with archive.open(name) as stream:\n                    for frame in pd.read_csv(\n                        stream,\n                        usecols=FIELDS,\n                        chunksize=100_000,\n                        dtype={"Tail_Number": "string"},\n                    ):\n                        active = frame.FlightDate.ge(earliest)\n                        targets.append(\n                            frame.loc[\n                                active & frame.Origin.isin(AIRPORTS), FIELDS\n                            ].copy()\n                        )\n                        inbound.append(\n                            frame.loc[active & frame.Dest.isin(AIRPORTS), FIELDS].copy()\n                        )\n            print(f"Read {year}-{month:02d}", flush=True)\n    with TemporaryDirectory(dir=ROOT) as temp:\n        for parts, name in [\n            (targets, "flights_raw.parquet"),\n            (inbound, "inbound_history.parquet"),\n        ]:\n            destination = Path(temp) / name\n            data = pd.concat(parts, ignore_index=True)\n            data.to_parquet(destination, index=False)\n            destination.replace(ROOT / name)\n            print(name, len(data))\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
    Path('pipeline/iem.py').write_text('"""Attach IEM airport observations to flights as of each prediction checkpoint."""\n\nimport time\nfrom datetime import UTC, datetime, timedelta\nfrom email.utils import parsedate_to_datetime\nfrom io import BytesIO\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport airportsdata\nimport pandas as pd\nimport requests\n\nCHECKPOINT_HOURS = [2, 6]\nMAX_OBSERVATION_AGE = pd.Timedelta(hours=3)\nREQUEST_PAUSE_SECONDS = 10\nREQUEST_ATTEMPTS = 3\n\nROOT = Path(__file__).resolve().parents[1] / "fresh_2025"\nFLIGHTS = ROOT / "flights_raw.parquet"\nWEATHER_CACHE = ROOT / "weather_raw"\nWEATHER_PARQUET = ROOT / "weather_raw.parquet"\nDOWNLOAD_LOG = ROOT / "download_log.txt"\nOUTPUT_CSV = ROOT / "output_with_weather.csv"\nOUTPUT_PARQUET = ROOT / "output_with_weather.parquet"\nIEM_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"\nIEM_FIELDS = (\n    "tmpf", "dwpf", "drct", "sknt", "gust", "vsby", "p01i", "skyc1", "skyl1", "wxcodes",\n)\nSCHEDULE_FIELDS = ("FlightDate", "Origin", "CRSDepTime")\nOUTCOME_FIELDS = {\n    "DepTime", "DepDelay", "DepDel15", "ArrTime", "ArrDelay", "Cancelled", "Diverted",\n}\nUTC_DTYPE = "datetime64[ns, UTC]"\n\n\nclass DownloadError(RuntimeError):\n    """IEM failed to return a valid CSV for the requested interval."""\n\n\ndef log_download(message):\n    timestamp = datetime.now(UTC).isoformat(timespec="seconds")\n    with DOWNLOAD_LOG.open("a", encoding="utf-8") as log:\n        print(f"{timestamp} {message}", file=log)\n\n\ndef read_station_csv(source, origin):\n    raw = pd.read_csv(source, dtype="string", keep_default_na=False, comment="#")\n    columns = ("station", "valid", *IEM_FIELDS)\n    if not set(columns).issubset(raw.columns):\n        raise ValueError(f"{origin}: expected an IEM CSV with {\', \'.join(columns)}")\n    raw = raw.loc[:, list(columns)]\n    if not raw["station"].isin((origin, f"K{origin}")).all():\n        raise ValueError(f"{origin}: unexpected station IDs {raw[\'station\'].unique().tolist()}")\n    raw["valid"] = pd.to_datetime(raw["valid"], utc=True, errors="coerce").astype(UTC_DTYPE)\n    if raw["valid"].isna().any():\n        raise ValueError(f"{origin}: IEM CSV contains unparseable observation timestamps")\n    return raw\n\n\ndef retry_after_seconds(response):\n    header = response.headers.get("Retry-After") if response is not None else None\n    if not header:\n        return 0\n    if header.isdecimal():\n        return int(header)\n    try:\n        return max(0, (parsedate_to_datetime(header) - datetime.now(UTC)).total_seconds())\n    except (TypeError, ValueError):\n        return 0\n\n\nclass IEMDownloader:\n    def __init__(self, session):\n        self.session = session\n        self.next_request = 0\n\n    def fetch(self, origin, station, start, end):\n        params = [\n            ("station", station),\n            *(("data", field) for field in IEM_FIELDS),\n            ("year1", start.year), ("month1", start.month), ("day1", start.day),\n            ("year2", end.year), ("month2", end.month), ("day2", end.day),\n            ("tz", "Etc/UTC"), ("format", "onlycomma"), ("latlon", "no"),\n            ("missing", "M"), ("trace", "T"), ("report_type", 3), ("report_type", 4),\n        ]\n        url = requests.Request("GET", IEM_URL, params=params).prepare().url\n        for attempt in range(REQUEST_ATTEMPTS):\n            delay = self.next_request - time.monotonic()\n            if delay > 0:\n                time.sleep(delay)\n            response = None\n            log_download(f"GET attempt={attempt + 1} {url}")\n            try:\n                response = self.session.get(url, timeout=(15, 120))\n                response.raise_for_status()\n                raw = read_station_csv(BytesIO(response.content), origin)\n            except (requests.RequestException, ValueError) as error:\n                log_download(f"FAILED station={station} {type(error).__name__}: {error}")\n                if attempt == REQUEST_ATTEMPTS - 1:\n                    self.next_request = time.monotonic() + max(\n                        REQUEST_PAUSE_SECONDS, retry_after_seconds(response)\n                    )\n                    raise DownloadError(f"{station} {start} through {end} (exclusive)") from error\n                print(f"{station}: request attempt {attempt + 1} failed ({error}); retrying.")\n                self.next_request = time.monotonic() + max(\n                    REQUEST_PAUSE_SECONDS * 2**attempt, retry_after_seconds(response)\n                )\n            else:\n                self.next_request = time.monotonic() + REQUEST_PAUSE_SECONDS\n                log_download(f"OK rows={len(raw)} {response.url}")\n                return response.content, raw\n\n    def fetch_range(self, origin, station, start, end):\n        try:\n            return self.fetch(origin, station, start, end)\n        except DownloadError:\n            first_boundary = (pd.Timestamp(start) + pd.offsets.MonthBegin()).date()\n            if first_boundary >= end:\n                raise\n\n        print(f"{station}: full-range download failed; requesting calendar months.")\n        frames = []\n        while start < end:\n            boundary = min((pd.Timestamp(start) + pd.offsets.MonthBegin()).date(), end)\n            _, raw = self.fetch(origin, station, start, boundary)\n            frames.append(raw)\n            start = boundary\n        raw = pd.concat(frames, ignore_index=True)\n        content = raw.to_csv(index=False, date_format="%Y-%m-%d %H:%M").encode("utf-8")\n        return content, raw\n\n\ndef load_weather(stations, start, end):\n    """Reuse the station CSVs cached for the flight dataset\'s date window."""\n    cache_dir = WEATHER_CACHE / f"{start}_{end}"\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    frames = []\n    with requests.Session() as session:\n        downloader = IEMDownloader(session)\n        for origin in stations:\n            cache = cache_dir / f"{origin}.csv"\n            try:\n                raw = read_station_csv(cache, origin)\n            except FileNotFoundError:\n                content, raw = downloader.fetch_range(origin, origin, start, end)\n                if raw.empty:\n                    print(f"{origin}: no observations; trying K{origin}.")\n                    content, raw = downloader.fetch_range(origin, f"K{origin}", start, end)\n                with TemporaryDirectory(dir=WEATHER_CACHE) as temp_dir:\n                    temporary = Path(temp_dir) / cache.name\n                    temporary.write_bytes(content)\n                    temporary.replace(cache)\n                print(f"{origin}: cached {len(raw):,} observations.")\n            else:\n                print(f"{origin}: loaded {len(raw):,} observations from cache.")\n            if raw.empty:\n                print(f"{origin}: no IEM observations available in the downloaded range.")\n            # Retain the IEM identifier and use the BTS airport code for the join.\n            frames.append(raw.assign(Origin=origin))\n\n    combined = pd.concat(frames, ignore_index=True)\n    in_range = combined["valid"].ge(pd.Timestamp(start, tz="UTC")) & combined["valid"].lt(\n        pd.Timestamp(end, tz="UTC")\n    )\n    return combined.loc[in_range].reset_index(drop=True)\n\n\ndef inspect_flights(flights):\n    print("Columns:", flights.columns.tolist())\n    print("Dtypes:")\n    print(flights.dtypes.to_string())\n    print("Head:")\n    print(flights.head().to_string(index=False))\n    print(f"Flight rows: {len(flights):,}")\n    if not set(SCHEDULE_FIELDS).issubset(flights.columns):\n        raise ValueError(f"Expected schedule columns {SCHEDULE_FIELDS}")\n\n    dates = pd.to_datetime(flights["FlightDate"], format="%Y-%m-%d", errors="coerce")\n    if dates.notna().sum() == 0:\n        raise ValueError("FlightDate contains no valid dates for the IEM request.")\n    first, last = dates.min().date(), dates.max().date()\n    print(f"FlightDate range: {first} through {last}")\n    stations = sorted(flights["Origin"].dropna().unique())\n    if not stations:\n        raise ValueError("Origin contains no airport codes for the IEM request.")\n    if any(len(code) != 3 or not code.isascii() or not code.isalnum() for code in stations):\n        raise ValueError(f"Expected three-character IATA origin codes; found {stations}")\n    print("Origin stations:", ", ".join(stations))\n    # IEM excludes its end date. Include the entire day after the last FlightDate.\n    start = first - timedelta(days=2)\n    end = last + timedelta(days=2)\n    print(f"IEM window (UTC): {start} through {end - timedelta(days=1)}, inclusive")\n    return stations, start, end\n\n\ndef scheduled_departures_utc(schedule):\n    # Only flight information known in advance can determine the checkpoint.\n    assert tuple(schedule.columns) == SCHEDULE_FIELDS\n    assert OUTCOME_FIELDS.isdisjoint(schedule.columns)\n\n    dates = pd.to_datetime(schedule["FlightDate"], format="%Y-%m-%d", errors="coerce")\n    hhmm = pd.to_numeric(schedule["CRSDepTime"], errors="coerce")\n    hours, minutes = hhmm // 100, hhmm % 100\n    valid = hhmm.ge(0) & hhmm.mod(1).eq(0) & (hhmm.lt(2400) | hhmm.eq(2400)) & minutes.lt(60)\n    # This treats e.g. 5 as 0005 and adds 24 hours for 2400.\n    local = dates + pd.to_timedelta((hours * 60 + minutes).where(valid), unit="min")\n\n    airports = airportsdata.load("IATA")\n    departure = pd.Series(pd.NaT, index=schedule.index, dtype=UTC_DTYPE, name="scheduled_dep_utc")\n    for origin, rows in schedule.groupby("Origin", sort=False).groups.items():\n        tz = airports.get(origin, {}).get("tz")\n        if not tz:\n            print(f"{origin}: no timezone available for {len(rows):,} rows.")\n            continue\n        # Use the first (DST) occurrence of the repeated fall-back hour at US airports.\n        # Advance a nonexistent spring-forward time to the first valid local time.\n        converted = local.loc[rows].dt.tz_localize(tz, ambiguous=True, nonexistent="shift_forward")\n        departure.loc[rows] = converted.dt.tz_convert("UTC").astype(UTC_DTYPE)\n    print(f"Scheduled departure UTC conversion failures: {departure.isna().sum():,} / {len(schedule):,}")\n    return departure\n\n\ndef build_weather_features(raw):\n    weather = raw.loc[:, ["Origin", "valid", *IEM_FIELDS]]\n    assert OUTCOME_FIELDS.isdisjoint(weather.columns)\n    features = weather.loc[:, ["Origin", "valid"]].copy()\n    missing = {"M": pd.NA, "": pd.NA}\n    for field in ("tmpf", "dwpf", "drct", "sknt", "gust", "vsby", "p01i", "skyl1"):\n        values = weather[field].replace(missing)\n        if field == "p01i":\n            # Keep the reported accumulation. IEM\'s numeric trace convention is 0.0001 inches.\n            values = values.replace("T", "0.0001")\n        features[f"wx_{field}"] = pd.to_numeric(values, errors="coerce").astype("float64")\n    features["wx_skyc1"] = weather["skyc1"].replace(missing)\n\n    for field in ("gust", "skyl1"):\n        values = features[f"wx_{field}"]\n        features[f"wx_{field}_missing"] = values.isna().astype("Int8")\n        features[f"wx_{field}_filled"] = values.fillna(0)\n\n    codes = weather["wxcodes"].replace(missing).str.upper()\n    # A missing code field leaves the indicators missing on that report.\n    for name, code in {"ts": "TS", "fg": "FG", "ra": "RA", "sn": "SN", "fz": "FZ", "br_hz": "BR|HZ"}.items():\n        # Match METAR\'s two-letter codes, including combinations such as -FZRA and +TSRA.\n        pattern = rf"(?:^|\\s)[+-]?(?:[A-Z]{{2}})*(?:{code})"\n        features[f"wx_{name}"] = codes.str.contains(pattern).astype("Int8")\n\n    return features\n\n\ndef attach_weather(flights, raw):\n    original_columns = flights.columns.tolist()\n    original_rows = len(flights)\n    schedule = flights.loc[:, list(SCHEDULE_FIELDS)]\n    departure = scheduled_departures_utc(schedule)\n    if "scheduled_dep_utc" in original_columns:\n        raise ValueError("Input already contains scheduled_dep_utc")\n\n    # merge_asof needs the time key globally sorted, even with a station grouping key.\n    left = pd.DataFrame({"Origin": schedule["Origin"].astype("string"), "departure": departure})\n    left = left.dropna(subset=["departure"]).sort_values("departure", kind="stable")\n    left = left.reset_index(names="flight_row")\n    # If IEM lists multiple reports at the same time, merge_asof uses the last listed report.\n    weather = build_weather_features(raw).sort_values("valid", kind="stable").reset_index(drop=True)\n    right = weather.loc[:, ["Origin", "valid"]].assign(weather_row=weather.index)\n    right["Origin"] = right["Origin"].astype("string")\n    values = weather.drop(columns=["Origin", "valid"])\n    flights["scheduled_dep_utc"] = departure\n\n    for hours in CHECKPOINT_HOURS:\n        left["T"] = left["departure"] - pd.Timedelta(hours=hours)\n        # Match only observations at or before T, at the same airport and within three hours.\n        # Join the weather row ID first to keep the time-key join small.\n        matches = pd.merge_asof(\n            left.loc[:, ["flight_row", "Origin", "T"]], right,\n            by="Origin", left_on="T", right_on="valid", direction="backward",\n            tolerance=MAX_OBSERVATION_AGE, allow_exact_matches=True,\n        )\n        ages = matches["T"] - matches["valid"]\n        if (ages.lt(pd.Timedelta(0)) | ages.gt(MAX_OBSERVATION_AGE)).any():\n            raise AssertionError(f"T-{hours}h: observation outside the allowed time window")\n\n        # Restore CSV row order, including rows with invalid schedules and unmatched weather.\n        matched = matches.set_index("flight_row").reindex(flights.index)\n        attached = values.reindex(matched["weather_row"].to_numpy()).set_axis(flights.index)\n        attached["wx_obs_time"] = matched["valid"]\n        attached["wx_obs_age_min"] = (matched["T"] - matched["valid"]).dt.total_seconds() / 60\n        attached = attached.add_suffix(f"_T{hours}h")\n        if set(attached.columns) & set(flights.columns):\n            raise ValueError(f"T-{hours}h: output feature names overlap existing columns")\n        for column in attached:\n            flights[column] = attached[column]\n\n        age_column = f"wx_obs_age_min_T{hours}h"\n        sample = flights.loc[:, ["Origin", "scheduled_dep_utc", f"wx_obs_time_T{hours}h", age_column]].head().copy()\n        sample.insert(2, "T", matched["T"].head())\n        print(f"\\nT-{hours}h timestamp audit sample:")\n        print(sample.to_string(index=False))\n        print(f"T-{hours}h flights with weather: {matched[\'valid\'].notna().sum():,} / {original_rows:,}")\n        print(f"T-{hours}h median weather age: {attached[age_column].median():.1f} min")\n        print(f"T-{hours}h missing by weather column:")\n        print(attached.isna().mean().mul(100).to_string(float_format=lambda value: f"{value:.3f}%"))\n\n    assert len(flights) == original_rows\n    assert flights.columns[:len(original_columns)].tolist() == original_columns\n    print(f"\\nFlight rows before / after: {original_rows:,} / {len(flights):,}")\n    print(f"New columns: {len(flights.columns) - len(original_columns)}")\n    return flights\n\n\ndef write_outputs(raw, flights):\n    # Stage the files so a failed write cannot leave a partially written CSV or Parquet.\n    with TemporaryDirectory(prefix=".weather-", dir=ROOT) as temp_dir:\n        staging = Path(temp_dir)\n        raw.to_parquet(staging / WEATHER_PARQUET.name, engine="pyarrow", index=False)\n        flights.to_parquet(staging / OUTPUT_PARQUET.name, engine="pyarrow", index=False)\n        flights.to_csv(\n            staging / OUTPUT_CSV.name, index=False, chunksize=50_000,\n            date_format="%Y-%m-%dT%H:%M:%SZ",\n        )\n        for output in (WEATHER_PARQUET, OUTPUT_PARQUET, OUTPUT_CSV):\n            (staging / output.name).replace(output)\n            print(f"Wrote {output.name}")\n\n\ndef main():\n    flights = pd.read_parquet(FLIGHTS)\n    stations, start, end = inspect_flights(flights)\n    raw = load_weather(stations, start, end)\n    print(f"IEM observations in the requested range: {len(raw):,}")\n    attach_weather(flights, raw)\n    print("Writing CSV and Parquet outputs...")\n    write_outputs(raw, flights)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
    Path('build_features.py').write_text('"""Create checkpoint-specific, chronological flight-delay modelling CSVs."""\n\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\n\nimport numpy as np\nimport pandas as pd\nfrom pandas.tseries.holiday import USFederalHolidayCalendar\n\nfrom pipeline import iem\n\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / "final_csv"\nWX = [\n    "skyc1",\n    "tmpf",\n    "dwpf",\n    "drct",\n    "sknt",\n    "gust",\n    "vsby",\n    "p01i",\n    "skyl1",\n    "obs_age_min",\n    "ts",\n    "fg",\n    "ra",\n    "sn",\n    "fz",\n    "br_hz",\n    "gust_missing",\n    "skyl1_missing",\n]\nSHARED = [\n    "month",\n    "dow",\n    "dep_hour",\n    "arr_hour",\n    "CRSElapsedTime",\n    "Distance",\n    "is_holiday",\n    "Origin",\n    "Dest",\n    "Reporting_Airline",\n]\nHISTORY = [\n    "n_departed_60min",\n    "prior_mean_delay_60min",\n    "late_rate_60min",\n    "n_scheduled_next_60min",\n    "has_history",\n]\nMETA = ["row_id", "FlightDate", "scheduled_dep_utc", "checkpoint_utc", "wx_obs_time"]\n\n\ndef activity(frame, checkpoint, allowed, protected, actual):\n    result = pd.DataFrame(index=frame.index, columns=HISTORY, dtype=float)\n    for origin, group in frame.groupby("Origin"):\n        rows = group.index\n        t = checkpoint.loc[rows]\n        departures = actual.loc[rows[allowed.loc[rows]]].sort_values()\n        delay = frame.DepDelay.loc[departures.index]\n        start = departures.searchsorted(t - pd.Timedelta(hours=1), side="right")\n        end = departures.searchsorted(t, side="right")\n        own = (\n            allowed.loc[rows]\n            & actual.loc[rows].gt(t - pd.Timedelta(hours=1))\n            & actual.loc[rows].le(t)\n        )\n        count = end - start - own.astype(int)\n        result.loc[rows, HISTORY[0]] = count\n        result.loc[rows, "has_history"] = count.gt(0).astype(int)\n        for col, values, own_values in [\n            (HISTORY[1], delay, group.DepDelay),\n            (HISTORY[2], delay.ge(15), group.DepDelay.ge(15)),\n        ]:\n            total = np.r_[0, values.to_numpy().cumsum()]\n            result.loc[rows, col] = (\n                total[end] - total[start] - own_values.astype(float).where(own, 0)\n            ) / count.replace(0, np.nan)\n        schedule = group.scheduled_dep_utc.sort_values()\n        result.loc[rows, HISTORY[3]] = schedule.searchsorted(\n            t + pd.Timedelta(hours=1), side="right"\n        ) - schedule.searchsorted(t, side="right")\n        # Withhold outcome-history fields when the window overlaps a sealed local date.\n        # A partial count would incorrectly imply that held-out events were observed.\n        blocked = protected.loc[rows]\n        if blocked.any():\n            zone = {\n                "ATL": "America/New_York",\n                "ORD": "America/Chicago",\n                "LAX": "America/Los_Angeles",\n            }[origin]\n            local = t.dt.tz_convert(zone)\n            sealed_dates = set(group.FlightDate[blocked])\n            unknown = local.dt.strftime("%Y-%m-%d").isin(sealed_dates) | (\n                (local - pd.Timedelta(hours=1))\n                .dt.strftime("%Y-%m-%d")\n                .isin(sealed_dates)\n            )\n            result.loc[rows[unknown], [HISTORY[i] for i in (0, 1, 2, 4)]] = np.nan\n    return result\n\n\ndef main():\n    frame = pd.read_parquet(iem.OUTPUT_PARQUET)\n    dates = frame.FlightDate\n    schedule = frame.scheduled_dep_utc\n    assert schedule.notna().all()\n    actual = schedule + pd.to_timedelta(frame.DepDelay, unit="min")\n    eligible = frame.Cancelled.eq(0) & frame.DepDelay.notna() & frame.DepDel15.notna()\n    assert (\n        frame.loc[eligible, "DepDelay"].ge(15).eq(frame.loc[eligible, "DepDel15"]).all()\n    )\n    old_test = frame.Origin.eq("ATL") & (\n        dates.between("2025-01-27", "2025-01-31")\n        | dates.between("2025-05-13", "2025-05-29")\n    )\n    test = dates.between("2025-12-01", "2025-12-31")\n    protected = old_test\n    allowed = eligible & ~protected\n    validation = eligible & dates.between("2025-09-01", "2025-11-28")\n    train = allowed & dates.between("2025-01-01", "2025-08-30")\n    # Label must have become observable before the earliest validation prediction.\n    train &= actual.lt((schedule[validation] - pd.Timedelta(hours=6)).min())\n    first_test_prediction = (schedule[test & eligible] - pd.Timedelta(hours=6)).min()\n    validation &= actual.lt(first_test_prediction)\n    masks = {"train": train, "validation": validation, "test": test & eligible}\n    assert not (train & validation | train & test | validation & test).any()\n    assert set(frame.Origin.unique()) == {"ATL", "ORD", "LAX"}\n    assert schedule[train].max() < (schedule[validation] - pd.Timedelta(hours=6)).min()\n    assert schedule[validation].max() < first_test_prediction\n    assert dates[test & eligible].min() == "2025-12-01"\n    assert dates[test & eligible].max() == "2025-12-31"\n    day = pd.to_datetime(dates)\n    shared = frame[\n        ["CRSElapsedTime", "Distance", "Origin", "Dest", "Reporting_Airline"]\n    ].copy()\n    shared["month"], shared["dow"] = day.dt.month, day.dt.dayofweek\n    shared["dep_hour"] = (frame.CRSDepTime // 100) % 24\n    shared["arr_hour"] = (frame.CRSArrTime // 100) % 24\n    holidays = USFederalHolidayCalendar().holidays(day.min(), day.max())\n    shared["is_holiday"] = day.isin(holidays).astype(int)\n    OUT.mkdir(exist_ok=True)\n    with TemporaryDirectory(dir=OUT) as temp:\n        staged = Path(temp)\n        for hours in (2, 6):\n            suffix = f"_T{hours}h"\n            t = schedule - pd.Timedelta(hours=hours)\n            obs = frame["wx_obs_time" + suffix]\n            age = (t - obs).dt.total_seconds() / 60\n            assert (\n                age.equals(frame["wx_obs_age_min" + suffix])\n                and age.dropna().between(0, 180).all()\n            )\n            wx = frame[["wx_" + s + suffix for s in WX]].copy()\n            wx["codes_known" + suffix] = frame["wx_ts" + suffix].notna().astype(int)\n            wx["wx_unmatched" + suffix] = obs.isna().astype(int)\n            history = activity(frame, t, allowed, protected, actual)\n            # Independent event-window audit on development rows, including empty windows.\n            for i in dates[train | validation].sample(300, random_state=42).index:\n                same = frame.Origin.eq(frame.at[i, "Origin"])\n                past = (\n                    allowed\n                    & same\n                    & actual.gt(t[i] - pd.Timedelta(hours=1))\n                    & actual.le(t[i])\n                )\n                past.at[i] = False\n                delays = frame.DepDelay[past]\n                demand = (\n                    same & schedule.gt(t[i]) & schedule.le(t[i] + pd.Timedelta(hours=1))\n                ).sum()\n                assert history.at[i, HISTORY[3]] == demand\n                if pd.notna(history.at[i, HISTORY[0]]):\n                    expected = [\n                        len(delays),\n                        delays.mean(),\n                        delays.ge(15).mean(),\n                        int(len(delays) > 0),\n                    ]\n                    np.testing.assert_allclose(\n                        history.loc[i, [HISTORY[k] for k in (0, 1, 2, 4)]],\n                        expected,\n                        equal_nan=True,\n                    )\n            inputs = pd.concat([shared[SHARED], wx, history.add_suffix(suffix)], axis=1)\n            assert inputs.shape[1] == 35\n            meta = pd.DataFrame(\n                {\n                    "row_id": frame.index,\n                    "FlightDate": dates,\n                    "scheduled_dep_utc": schedule,\n                    "checkpoint_utc": t,\n                    "wx_obs_time": obs,\n                }\n            )\n            table = pd.concat([meta, inputs, frame.DepDel15.rename("LATE")], axis=1)\n            for name, mask in masks.items():\n                part = table.loc[mask].sort_values(["checkpoint_utc", "row_id"])\n                part.to_csv(\n                    staged / f"{name}{suffix}.csv", index=False, chunksize=30000\n                )\n                print(\n                    name,\n                    suffix,\n                    len(part),\n                    part.FlightDate.min(),\n                    part.FlightDate.max(),\n                    flush=True,\n                )\n        for path in staged.glob("*.csv"):\n            path.replace(OUT / path.name)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
    !python pipeline/bts.py && python pipeline/iem.py && python build_features.py
else:
    print('Found existing dataset files!')


In [ ]:
# 3. Run Person 2 benchmarks via shared train.py
!rm -rf runs/trees runs/pca
!python train.py --models decision_tree random_forest adaboost --out runs/trees
!python train.py --models logreg pca_logreg --out runs/pca

# Optional fast tuning example on 100,000 rows:
# !rm -rf runs/dt8 && python train.py --models decision_tree --params '{"max_depth":8,"ccp_alpha":0.0001}' --train-limit 100000 --out runs/dt8


In [ ]:
# 4. Inspect results & zip outputs for Person 4
import pandas as pd
print('=== Trees ===')
display(pd.read_csv('runs/trees/results.csv'))
print('=== PCA Comparison ===')
display(pd.read_csv('runs/pca/results.csv'))
!zip -rq person2_runs.zip runs/trees runs/pca
print('Packaged person2_runs.zip for Person 4!')
